# 07 Transition-Event Diagnostics
Question: What do the current strategy returns look like around real Part I macro regime transition dates?

**Warning:** This notebook is research-only transition diagnostics and must not be used for forecasting, signal generation, ranking, optimization, allocation, dashboarding, or trading decisions.

This notebook uses the checked-in real Part I transition table together with the current strategy return input. The purpose is to inspect descriptive transition coverage and event-window diagnostics only.


## Load Real Inputs And Reuse Locked Validations
Load the real macro regime file, the real transition table, and the current strategy return input. Reuse the existing loader and validation path before showing any descriptive transition diagnostics.


In [ ]:
from pathlib import Path
import sys

def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing 'src' and 'data'.")

REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.conditioning.transition_diagnostics_report import build_notebook_07_tables
from src.data.loaders import load_regimes, load_strategy_returns, load_transitions
from src.data.validators import run_all_validations

regimes = load_regimes(REPO_ROOT / "data" / "macro" / "regimes.csv")
transitions = load_transitions(REPO_ROOT / "data" / "macro" / "transitions.csv")
strategy_returns = load_strategy_returns(REPO_ROOT / "data" / "strategies" / "strategy_returns_example.csv")

run_all_validations(regimes, transitions, strategy_returns)

tables = build_notebook_07_tables(regimes, transitions, strategy_returns, window=3)
display(tables["integration_overview"])
display(tables["transition_coverage_summary"])


## Transition Coverage Diagnostics
Report how many real Part I transition rows fall inside the current strategy date range versus outside it. A zero-count for aligned transitions is acceptable when the current strategy history does not cover transition months and is not a contract failure. When aligned transitions do exist, the notebook distinguishes complete event windows from incomplete ones, including expected strategy windows with zero observations anywhere in the event window.


In [ ]:
display(tables["transition_window_status"])
display(tables["transition_window_completeness_summary"])
display(tables["aligned_transitions"])


## Outside-Coverage Transition Rows
This compact table lists the real transition dates that fall outside the current strategy history. The current strategy window explains why these rows are skipped from event-window diagnostics.


In [ ]:
display(tables["outside_coverage_transitions"])


## Transition Event Windows
Use the shared transition-window functions to inspect per-strategy event windows around each aligned transition. The diagnostics table is built on the expected strategy-by-transition grid, so zero-observation windows are reported explicitly even when the raw event-window table has no rows for that strategy/transition pair. The aggregate transition summaries below use only complete per-strategy windows and report incomplete-window counts separately.


In [ ]:
display(tables["transition_window_diagnostics"])
display(tables["transition_windows"])
display(tables["transition_window_summary"])
display(tables["transition_cumulative_profile"])
